# Fine-tuning Whisper tiny for Indian-accented English

This notebook fine-tunes `openai/whisper-tiny` on Indian-accented English and measures the change on
the same 120 Speech Accent Archive clips used in our paper (test set A) and on Svarah (test set B).

**Data**
- Indian English (training): about 150 hours of NPTEL2020 lecture speech (CC BY 4.0), read from the
  `skbose/indian-english-nptel-v0` mirror on Hugging Face, spread evenly over about 450 speakers.
- American English (training): about 15 hours of LibriSpeech `train.clean.100` (CC BY 4.0), about
  10% of the mix.
- Test set A: the 120 clips in this repository. Never trained on.
- Test set B: Svarah (AI4Bharat, CC BY 4.0). Never trained on.

**Time plan (hard cap 10 hours)**

| Stage | Cells | Budget | Expected |
|---|---|---|---|
| Setup | 1 to 4 | | about 5 min |
| 1. Data download and prep | 5 | 90 min | 30 to 50 min |
| 2. Baseline testing | 6 to 7 | 20 min | 10 to 15 min |
| 3. Training | 8 to 10 | 7 h | about 1.5 to 2.5 h (5 epochs; cell 9 measures it) |
| 4. Final testing and summary | 11 to 13 | 60 min | 15 to 25 min |

Cell 9 times the first 50 training steps and sets the total step count so training fits 7 hours,
capped at `MAX_EPOCHS` passes over the data. Every stage logs its own time to
`stage_times.json` on Drive, and cell 13 prints planned versus actual time.

**After a disconnect:** reconnect to a T4 runtime and run all cells from the top again. Cell 5 rebuilds
the same training set from the manifest saved on Drive, cells 6, 7 and 9 skip work that already
finished, and cell 10 resumes from the last checkpoint on Drive. Checkpoints are saved at least every
30 minutes, so a disconnect loses at most about 30 minutes of training.

## Setup

In [ ]:
# Cell 1. Hugging Face login, Google Drive, GPU check
HF_TOKEN = "PASTE_YOUR_HF_READ_TOKEN_HERE"
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN") or HF_TOKEN
except Exception:
    pass
from huggingface_hub import login
login(token=HF_TOKEN)
del HF_TOKEN

from google.colab import drive
drive.mount("/content/drive")

import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "No GPU found: switch the runtime to T4 GPU.")

In [ ]:
# Cell 2. Packages (versions match the environment that produced the paper's numbers)
!pip -q install openai-whisper==20250625 jiwer==4.0.0 num2words

In [ ]:
# Cell 3. Configuration, repository clone, stage timer
import os, io, re, gc, json, time, math, glob, random, shutil, subprocess, zipfile
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, torch

DRIVE_DIR = Path("/content/drive/MyDrive/ASR_finetune")   # everything that must survive a disconnect
WORK      = Path("/content/work")                         # fast local disk, rebuilt after a disconnect
REPO_DIR  = Path("/content/ASR_unicorn_labs")
RESULTS   = DRIVE_DIR / "results"
CKPT_DIR  = DRIVE_DIR / "checkpoints"
FINAL_DIR = DRIVE_DIR / "final_model"
for p in (DRIVE_DIR, WORK, RESULTS, CKPT_DIR):
    p.mkdir(parents=True, exist_ok=True)

MODEL_ID    = "openai/whisper-tiny"
NPTEL_REPO  = "skbose/indian-english-nptel-v0"
LIBRI_REPO  = "openslr/librispeech_asr"
SVARAH_REPO = "ai4bharat/Svarah"

# Data
NPTEL_HOURS      = 150     # Indian English training hours
LIBRI_HOURS      = 15      # American English training hours (about 10% of the mix)
LIBRI_SHARE      = LIBRI_HOURS / (NPTEL_HOURS + LIBRI_HOURS)   # kept if hours are scaled down
LIBRI_SHARDS     = 8       # LibriSpeech train.clean.100 shards to sample from (about 7 h each)
CANDIDATE_FACTOR = 1.12    # read 12% more clean NPTEL audio than needed, to cover the rate filter
NPTEL_STOP_MIN   = 65      # stop downloading NPTEL shards after this many minutes of prep
NPTEL_GB_PER_HOUR = 0.115  # measured size of NPTEL audio on disk
DISK_MARGIN_GB   = 10      # free disk to keep in reserve; hours scale down if this is not met
MIN_SEC, MAX_SEC = 2.0, 30.0
MIN_WORDS        = 4
MIN_WPS, MAX_WPS = 1.0, 4.5   # fixed words-per-second limits for all training data
WPS_ROBUST_K     = 2.5     # NPTEL only: also drop rates beyond median +/- 2.5 robust SD of the data
N_VAL_NPTEL, N_VAL_LIBRI = 200, 100   # held-out clips used only to pick the best checkpoint

# Training
LR, WARMUP_STEPS = 1e-5, 500
BATCH_SIZE       = 32
TRAIN_BUDGET_H   = 7.0
MAX_EPOCHS       = 5       # upper limit on passes over the training data
SAVE_EVERY_MIN   = 30
EVAL_EVERY_STEPS = 1000
SEED             = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Arnavjagetiya/ASR_unicorn_labs", str(REPO_DIR)], check=True)

TIME_LOG = DRIVE_DIR / "stage_times.json"
PLAN_MIN = {"1 data prep": 90, "2 baseline": 20, "3 training": 420, "4 final testing": 60}

class stage:
    """Times a block and adds the minutes to stage_times.json on Drive (reruns add up)."""
    def __init__(self, name): self.name = name
    def __enter__(self):
        self.t0 = time.time(); print(f"[{self.name}] started {time.strftime('%H:%M:%S')}"); return self
    def __exit__(self, *exc):
        mins = (time.time() - self.t0) / 60
        log = json.loads(TIME_LOG.read_text()) if TIME_LOG.exists() else {}
        log.setdefault(self.name, []).append(round(mins, 2))
        TIME_LOG.write_text(json.dumps(log, indent=1))
        print(f"[{self.name}] took {mins:.1f} min (all runs: {sum(log[self.name]):.1f} of {PLAN_MIN.get(self.name, '?')} planned)")

def budget_report():
    log = json.loads(TIME_LOG.read_text()) if TIME_LOG.exists() else {}
    rows = [(k, PLAN_MIN[k], round(sum(log.get(k, [])), 1)) for k in PLAN_MIN]
    df = pd.DataFrame(rows, columns=["stage", "planned_min", "used_min"])
    print(df.to_string(index=False))
    print(f"Total used: {df.used_min.sum():.0f} of 600 min")
    return df

print("Device:", DEVICE, "| test clips:", len(list((REPO_DIR / "india_clips").glob("*.mp3"))),
      "Indian,", len(list((REPO_DIR / "us_clips").glob("*.mp3"))), "US")

In [ ]:
# Cell 4. Scoring (copied unchanged from make_figures.py so numbers match the paper) and shared helpers
import jiwer, whisper, soundfile as sf, librosa
from scipy import stats
from num2words import num2words

_PUNCT = re.compile(r"[^a-z0-9'\s]")
_SPACES = re.compile(r"\s+")
NUM_WORDS = {
    "0": "zero", "1": "one", "2": "two", "3": "three", "4": "four",
    "5": "five", "6": "six", "7": "seven", "8": "eight", "9": "nine",
    "10": "ten", "11": "eleven", "12": "twelve", "13": "thirteen",
    "14": "fourteen", "15": "fifteen", "16": "sixteen", "17": "seventeen",
    "18": "eighteen", "19": "nineteen", "20": "twenty",
}

def normalise(text):
    """Lowercase, punctuation to space, apostrophes kept, spaces collapsed."""
    return _SPACES.sub(" ", _PUNCT.sub(" ", str(text).lower())).strip()

def spell_numbers(text):
    """Map standalone numerals back to words so orthography is not penalised."""
    return " ".join(NUM_WORDS.get(tok, tok) for tok in text.split())

def has_digit(text):
    return any(tok.isdigit() for tok in text.split())

def score(df, spelled=False):
    """Add per-clip WER and error counts. Returns the same frame."""
    rows = []
    for ref, hyp in zip(df["ref"], df["hyp"]):
        if spelled:
            ref, hyp = spell_numbers(ref), spell_numbers(hyp)
        out = jiwer.process_words(ref, hyp)
        rows.append((out.wer, out.substitutions, out.deletions, out.insertions))
    df[["wer", "subs", "dels", "ins"]] = pd.DataFrame(rows, index=df.index)
    return df

def cohens_d(a, b):
    na, nb = len(a), len(b)
    pooled = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))
    return (a.mean() - b.mean()) / pooled if pooled else np.nan

def summarise(df, condition):
    """One row per group plus the between-group comparison."""
    india = df.loc[df.group == "India", "wer"].to_numpy()
    us = df.loc[df.group == "US", "wer"].to_numpy()
    p = stats.mannwhitneyu(india, us, alternative="greater").pvalue
    out = {"condition": condition, "p_one_sided": p, "cohens_d": cohens_d(india, us),
           "abs_gap_pp": (india.mean() - us.mean()) * 100,
           "ratio": india.mean() / us.mean() if us.mean() else np.nan}
    for name, arr in (("us", us), ("india", india)):
        out[f"{name}_n"] = len(arr)
        out[f"{name}_mean_pct"] = arr.mean() * 100
        out[f"{name}_se_pct"] = arr.std(ddof=1) / np.sqrt(len(arr)) * 100
        out[f"{name}_median_pct"] = np.median(arr) * 100
        out[f"{name}_p90_pct"] = np.percentile(arr, 90) * 100
        out[f"{name}_max_pct"] = arr.max() * 100
        out[f"{name}_zero_error"] = int((arr == 0).sum())
    for name, grp in (("us", "US"), ("india", "India")):
        sub = df[df.group == grp]
        out[f"{name}_subs_total"] = int(sub["subs"].sum())
        out[f"{name}_dels_total"] = int(sub["dels"].sum())
        out[f"{name}_ins_total"] = int(sub["ins"].sum())
        out[f"{name}_clips_with_digits"] = int(sub["hyp"].map(has_digit).sum())
    return out

def substitution_pairs(df):
    """Count (reference word -> hypothesis word) substitutions per group."""
    counts = {g: Counter() for g in ["US", "India"]}
    for row in df.itertuples():
        out = jiwer.process_words(row.ref, row.hyp)
        ref_words, hyp_words = out.references[0], out.hypotheses[0]
        for chunk in out.alignments[0]:
            if chunk.type != "substitute":
                continue
            for i, j in zip(range(chunk.ref_start_idx, chunk.ref_end_idx),
                            range(chunk.hyp_start_idx, chunk.hyp_end_idx)):
                counts[row.group][(ref_words[i], hyp_words[j])] += 1
    return counts

# ---- test set A (same passage, file order and CSV columns as whisper_final.py)
GROUND_TRUTH = (
    "Please call Stella. Ask her to bring these things with her from the store: "
    "Six spoons of fresh snow peas, five thick slabs of blue cheese, and maybe a "
    "snack for her brother Bob. We also need a small plastic snake and a big toy "
    "frog for the kids. She can scoop these things into three red bags, and we "
    "will go meet her Wednesday at the train station."
)
TEST_A = ([("US", p) for p in sorted((REPO_DIR / "us_clips").glob("*.mp3"))]
          + [("India", p) for p in sorted((REPO_DIR / "india_clips").glob("*.mp3"))])

# the paper's own whisper-tiny transcripts (repository root, or Results/ in newer layouts)
PAPER_CSV = next(p for p in (REPO_DIR / "results_comparison_tiny.csv", REPO_DIR / "Results" / "results_comparison_tiny.csv")
                 if p.exists())

def load_scored(path_or_df, spelled=False):
    df = pd.read_csv(path_or_df) if not isinstance(path_or_df, pd.DataFrame) else path_or_df.copy()
    df["prediction"] = df["prediction"].fillna("")
    df["ref"] = df["ground_truth"].map(normalise)
    df["hyp"] = df["prediction"].map(normalise)
    return score(df, spelled=spelled)

def baseline_note():
    """One sentence explaining why the seeded rerun differs slightly from the paper's CSV."""
    paper, ours = load_scored(PAPER_CSV), load_scored(RESULTS / "testA_baseline_openai.csv")
    both = paper.merge(ours, on=["group", "file"], suffixes=("_paper", "_ours"))
    same = int((both.prediction_paper.str.strip() == both.prediction_ours.str.strip()).sum())
    p, o = summarise(paper, "paper"), summarise(ours, "ours")
    return (f"We use our seeded rerun of the unchanged whisper-tiny as the official baseline "
            f"({o['us_mean_pct']:.2f}% American, {o['india_mean_pct']:.2f}% Indian), which differs slightly from "
            f"the paper's {p['us_mean_pct']:.2f}% and {p['india_mean_pct']:.2f}% because openai-whisper switches to "
            f"random sampling when its first decoding attempt fails a quality check and the paper's run did not "
            f"fix the random seed, so {len(both) - same} of {len(both)} transcripts came out differently.")

def show_summary(rows):
    cols = ["condition", "us_mean_pct", "us_se_pct", "india_mean_pct", "india_se_pct",
            "abs_gap_pp", "ratio", "p_one_sided", "cohens_d", "us_ins_total", "india_ins_total"]
    print(pd.DataFrame(rows)[cols].round(4).to_string(index=False))

# ---- text and audio helpers
def train_text(text):
    """Paper normalisation, then any numerals written out as words."""
    words = [num2words(int(t)) if t.isdigit() else t for t in normalise(text).split()]
    return normalise(" ".join(words))

def decode_audio(b):
    """Audio bytes to 16 kHz mono float32."""
    try:
        y, sr = sf.read(io.BytesIO(b), dtype="float32", always_2d=False)
    except Exception:
        y, sr = librosa.load(io.BytesIO(b), sr=None, mono=False)
    if y.ndim > 1:
        y = y.mean(axis=1) if y.shape[1] < y.shape[0] else y.mean(axis=0)
    if sr != 16000:
        y = librosa.resample(y, orig_sr=sr, target_sr=16000)
    return y.astype(np.float32)

def audio_seconds(b):
    with sf.SoundFile(io.BytesIO(b)) as f:
        return f.frames / f.samplerate

# ---- the two transcription paths
def transcribe_openai(model, tag, **kw):
    """Paper path: openai-whisper transcribe() in full precision, exactly as whisper_final.py."""
    rows = []
    for group, path in TEST_A:
        torch.manual_seed(0)   # transcribe() samples at temperature > 0 when greedy output fails its checks
        text = model.transcribe(str(path), fp16=False, **kw)["text"].strip()
        rows.append({"group": group, "file": path.name, "ground_truth": GROUND_TRUTH,
                     "prediction": text, "model": tag})
    return pd.DataFrame(rows)

def make_asr_pipeline(model, processor):
    from transformers import pipeline
    return pipeline("automatic-speech-recognition", model=model, tokenizer=processor.tokenizer,
                    feature_extractor=processor.feature_extractor, chunk_length_s=30,
                    device=DEVICE, ignore_warning=True)

def transcribe_hf(asr, audios, batch_size=32):
    """Transformers path: English forced, task transcribe, 30 s chunks for longer clips."""
    outs = asr(audios, batch_size=batch_size,
               generate_kwargs={"language": "en", "task": "transcribe"})
    return [o["text"].strip() for o in outs]

def transcribe_hf_test_a(asr, tag):
    audios = [whisper.load_audio(str(p)) for _, p in TEST_A]
    preds = transcribe_hf(asr, audios, batch_size=16)
    return pd.DataFrame([{"group": g, "file": p.name, "ground_truth": GROUND_TRUTH,
                          "prediction": t, "model": tag} for (g, p), t in zip(TEST_A, preds)])

# ---- Hugging Face weights to openai-whisper format, so the fine-tuned model runs the paper path
def hf_to_openai(sd):
    out = {}
    for k, v in sd.items():
        if k == "proj_out.weight":
            continue
        k = k.removeprefix("model.")
        k = k.replace("encoder.layers.", "encoder.blocks.").replace("decoder.layers.", "decoder.blocks.")
        for a, b in [(".self_attn.q_proj", ".attn.query"), (".self_attn.k_proj", ".attn.key"),
                     (".self_attn.v_proj", ".attn.value"), (".self_attn.out_proj", ".attn.out"),
                     (".encoder_attn.q_proj", ".cross_attn.query"), (".encoder_attn.k_proj", ".cross_attn.key"),
                     (".encoder_attn.v_proj", ".cross_attn.value"), (".encoder_attn.out_proj", ".cross_attn.out"),
                     (".self_attn_layer_norm", ".attn_ln"), (".encoder_attn_layer_norm", ".cross_attn_ln"),
                     (".final_layer_norm", ".mlp_ln"), (".fc1", ".mlp.0"), (".fc2", ".mlp.2")]:
            k = k.replace(a, b)
        k = {"encoder.layer_norm.weight": "encoder.ln_post.weight",
             "encoder.layer_norm.bias": "encoder.ln_post.bias",
             "decoder.layer_norm.weight": "decoder.ln.weight",
             "decoder.layer_norm.bias": "decoder.ln.bias",
             "encoder.embed_positions.weight": "encoder.positional_embedding",
             "decoder.embed_positions.weight": "decoder.positional_embedding",
             "decoder.embed_tokens.weight": "decoder.token_embedding.weight"}.get(k, k)
        out[k] = v.detach().float().cpu()
    return out

# ---- Svarah (test set B); SVARAH_DIR and SV_TEXT are set in cell 5, processor in cells 6 and 8
def load_svarah():
    files = sorted(SVARAH_DIR.glob("data/*.parquet"))
    tables = [pq.read_table(f) for f in files]
    meta_cols = [c for c in tables[0].column_names if c not in ("audio",)]
    rows = []
    for f, t in zip(files, tables):
        audio = t.column("audio").combine_chunks().field("bytes").to_pylist()
        meta = t.select(meta_cols).to_pylist()
        for i, (b, m) in enumerate(zip(audio, meta)):
            rows.append({"clip_id": f"{f.stem}_{i:05d}", "audio_bytes": b, **m})
    df = pd.DataFrame(rows)
    df = df[df[SV_TEXT].map(normalise).str.len() > 0].reset_index(drop=True)
    return df

def run_svarah(model, tag, out_csv):
    if out_csv.exists():
        return pd.read_csv(out_csv)
    sv = load_svarah()
    audios = [decode_audio(b) for b in sv.audio_bytes]
    preds = transcribe_hf(make_asr_pipeline(model, processor), audios, batch_size=32)
    df = sv.drop(columns=["audio_bytes"]).rename(columns={SV_TEXT: "ground_truth"})
    df["prediction"], df["model"], df["group"] = preds, tag, "India"
    df.to_csv(out_csv, index=False)
    del audios; gc.collect()
    return df

print("Helpers ready.")

## Stage 1. Data download and prep (budget 90 min)

Cell 5 downloads Svarah, LibriSpeech and NPTEL shards and then cleans and selects the training data.
- **Clean-segment filter:** clips from 2 to 30 seconds, at least 4 words, and 1.0 to 4.5 words per
  second. A transcript outside that rate usually does not match its audio. We also drop transcripts
  with a word repeated four or more times in a row.
- **Tighter rate filter for NPTEL:** because about 14.5% of NPTEL transcript words are wrong, we also
  drop NPTEL clips whose speaking rate lies more than 2.5 robust standard deviations from the
  dataset's median. On a four-shard sample this band is about 1.0 to 4.1 words per second and removes
  about 3.7% of clips, compared with 3.0% for the fixed limits alone. The report gives the exact share.
- **No leakage:** we drop any training transcript that contains "stella" or shares two or more
  4-word sequences with the test passage, and any transcript that exactly matches a Svarah sentence.
- **Speaker spread:** NPTEL has about 450 distinct lecturers, and we see nearly all of them. We select
  clips round-robin across speakers, so each speaker contributes one clip per round until we reach
  the target hours. This spreads the audio evenly instead of drawing more from a few people.
- **Numbers as words:** we write any numerals in training text as words.

About 26 GB is downloaded and the peak local disk use is about 25 GB. If free disk is short, the cell
scales the hours down before downloading. NPTEL downloading also stops after `NPTEL_STOP_MIN` minutes.
In either case, training uses the hours collected so far, LibriSpeech is trimmed to stay at about 10%
of the mix, and the report says so. The manifest goes to Drive, so a
rerun after a disconnect rebuilds the same set.

In [ ]:
# Cell 5. Data download, cleaning, leakage checks, speaker-balanced selection
import pyarrow as pa, pyarrow.parquet as pq
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from concurrent.futures import ThreadPoolExecutor

MANIFEST = DRIVE_DIR / "train_manifest.csv"
PREP_REPORT = DRIVE_DIR / "prep_report.json"
RAW, CAND, TRAIN_DIR, VAL_DIR, SVARAH_DIR = (WORK / d for d in ("raw", "cand", "train", "val", "svarah"))
for d in (RAW, CAND, TRAIN_DIR, VAL_DIR):
    shutil.rmtree(d, ignore_errors=True); d.mkdir(parents=True)

api = HfApi()
def repo_files(repo, prefix):
    return sorted(f for f in api.list_repo_files(repo, repo_type="dataset")
                  if f.startswith(prefix) and f.endswith(".parquet"))

PASSAGE = normalise(GROUND_TRUTH).split()
def ngrams(words, n=4):
    return {tuple(words[i:i + n]) for i in range(len(words) - n + 1)}
PASSAGE_4GRAMS = ngrams(PASSAGE)

def reject_reason(text, dur, svarah_texts):
    words = text.split()
    if not (MIN_SEC <= dur <= MAX_SEC): return "duration"
    if len(words) < MIN_WORDS: return "too_few_words"
    if not (MIN_WPS <= len(words) / dur <= MAX_WPS): return "words_per_second_fixed"
    if re.search(r"\b(\w+)(?: \1\b){3,}", text): return "repeated_word"
    if "stella" in words or len(PASSAGE_4GRAMS & ngrams(words)) >= 2: return "test_passage_leak"
    if len(words) >= 5 and text in svarah_texts: return "svarah_leak"
    return None

def read_rows(path, text_col, speaker_col, id_col, source, shard):
    t = pq.read_table(path, columns=["audio", text_col, speaker_col, id_col])
    audio = t.column("audio").combine_chunks().field("bytes").to_pylist()
    return [{"audio_bytes": b, "raw_text": txt, "speaker": f"{source}:{spk}", "file_name": str(fid),
             "source": source, "shard": shard}
            for b, txt, spk, fid in zip(audio, t.column(text_col).to_pylist(),
                                         t.column(speaker_col).to_pylist(), t.column(id_col).to_pylist())]

def clean_rows(rows, svarah_texts, reasons, keep_ids=None, wps_log=None):
    """Applies the fixed filters. wps_log collects the speaking rate of every clip with a usable
    length and word count, so the dataset band can be set from the full distribution."""
    kept = []
    for r in rows:
        if keep_ids is not None and r["file_name"] not in keep_ids:
            continue
        r["text"] = train_text(r.pop("raw_text"))
        r["duration"] = audio_seconds(r["audio_bytes"])
        r["wps"] = len(r["text"].split()) / r["duration"] if r["duration"] else 0.0
        why = reject_reason(r["text"], r["duration"], svarah_texts)
        if wps_log is not None and why not in ("duration", "too_few_words"):
            wps_log.append(r["wps"])
        reasons["read"] += 1
        reasons[why or "passed_fixed_filters"] += 1
        if why is None or keep_ids is not None:
            kept.append(r)
    return kept

COLS = ["audio_bytes", "text", "speaker", "file_name", "source", "shard", "duration", "wps"]
META = ["file_name", "speaker", "duration", "wps", "shard", "source"]

def write_parquet(rows, path):
    pq.write_table(pa.table({c: [r[c] for r in rows] for c in COLS}), path)

def write_arrow(table, path):
    """Arrow stream file: datasets memory-maps it directly, so there is no second copy on disk."""
    with pa.OSFile(str(path), "wb") as sink, pa.ipc.new_stream(sink, table.schema) as w:
        w.write_table(table, max_chunksize=1000)

def round_robin(meta, hours):
    """Take one clip per speaker per round (random order) until the hour target is reached."""
    rng = np.random.default_rng(SEED)
    meta = meta.assign(r=rng.random(len(meta))).sort_values(["speaker", "r"])
    meta["rank"] = meta.groupby("speaker").cumcount()
    meta = meta.sort_values(["rank", "r"])
    return meta[meta.duration.cumsum() / 3600 <= hours].drop(columns=["r", "rank"])

def collect(repo, queue, text_col, spk_col, id_col, source, target_h, reasons, keep, wps_log, stop_min):
    """Downloads shards four at a time, cleans each, and stores the candidates on local disk."""
    fetch = lambda f: hf_hub_download(repo, f, repo_type="dataset", local_dir=str(RAW))
    pool = ThreadPoolExecutor(max_workers=4)
    futures = {f: pool.submit(fetch, f) for f in queue[:4]}
    meta, hours, stopped = [], 0.0, False
    for i, f in enumerate(queue):
        if keep is None and hours >= target_h:
            break
        if keep is None and stop_min and (time.time() - t_start) / 60 > stop_min:
            stopped = True
            print(f"{source}: download time limit reached; using the audio collected so far.")
            break
        if i + 4 < len(queue):
            futures[queue[i + 4]] = pool.submit(fetch, queue[i + 4])
        path = futures.pop(f).result()
        rows = clean_rows(read_rows(path, text_col, spk_col, id_col, source, f), svarah_texts,
                          reasons, keep, wps_log)
        os.remove(path)
        write_parquet(rows, CAND / f"{source}_{i:03d}.parquet")
        meta += [{k: r[k] for k in META} for r in rows]
        hours += sum(r["duration"] for r in rows) / 3600
        del rows; gc.collect()
        print(f"  {source} shard {i + 1}: {hours:.1f} h candidates, {len({m['speaker'] for m in meta})} speakers, "
              f"{(time.time() - t_start) / 60:.1f} min, {shutil.disk_usage(str(WORK)).free / 1e9:.0f} GB free")
    pool.shutdown(wait=False, cancel_futures=True)
    for fut in futures.values():   # remove shards that were prefetched but not used
        if fut.done() and not fut.exception():
            Path(fut.result()).unlink(missing_ok=True)
    return pd.DataFrame(meta, columns=META), stopped

with stage("1 data prep"):
    t_start = time.time()
    rebuild = MANIFEST.exists()
    manifest = pd.read_csv(MANIFEST, dtype={"file_name": str}) if rebuild else None
    keep = set(manifest.file_name) if rebuild else None
    print("Mode:", "rebuild from saved manifest" if rebuild else "first build")

    # -- Disk check: candidates for every hour we read, plus LibriSpeech, Svarah and prefetched shards
    free_gb = shutil.disk_usage(str(WORK)).free / 1e9
    fixed_gb = LIBRI_SHARDS * 0.5 + 6          # LibriSpeech shards, Svarah, validation, prefetch, models
    need_gb = NPTEL_HOURS * CANDIDATE_FACTOR * NPTEL_GB_PER_HOUR + fixed_gb
    print(f"Disk: {free_gb:.0f} GB free, about {need_gb:.0f} GB needed at peak")
    if not rebuild and need_gb > free_gb - DISK_MARGIN_GB:
        NPTEL_HOURS = max(10, int((free_gb - DISK_MARGIN_GB - fixed_gb) / (CANDIDATE_FACTOR * NPTEL_GB_PER_HOUR)))
        LIBRI_HOURS = round(NPTEL_HOURS * LIBRI_SHARE / (1 - LIBRI_SHARE), 1)
        print(f"Not enough disk: scaling down to {NPTEL_HOURS} h NPTEL and {LIBRI_HOURS} h LibriSpeech")

    # -- Svarah (test set B): download, and collect its sentences for the leakage check
    snapshot_download(SVARAH_REPO, repo_type="dataset", allow_patterns=["data/*.parquet", "README.md"],
                      local_dir=str(SVARAH_DIR))
    sv_files = sorted(SVARAH_DIR.glob("data/*.parquet"))
    sv_schema = pq.read_schema(sv_files[0])
    print("Svarah columns:", sv_schema.names)
    SV_TEXT = next(c for c in ("text", "sentence", "transcript", "transcription") if c in sv_schema.names)
    svarah_texts = set()
    for f in sv_files:
        svarah_texts |= {normalise(x) for x in pq.read_table(f, columns=[SV_TEXT]).column(SV_TEXT).to_pylist()}
    print(f"Svarah: {len(svarah_texts)} distinct sentences, text column '{SV_TEXT}'")

    # -- Validation clips (only used to pick the best checkpoint)
    val = []
    for repo, prefix, cols, src, n in [
            (LIBRI_REPO, "clean/validation/", ("text", "speaker_id", "id"), "libri", N_VAL_LIBRI),
            (NPTEL_REPO, "data/test-", ("transcription_normalised", "speaker_name", "file_name"), "nptel", N_VAL_NPTEL)]:
        p = hf_hub_download(repo, repo_files(repo, prefix)[0], repo_type="dataset", local_dir=str(RAW))
        v = clean_rows(read_rows(p, *cols, src, "validation"), svarah_texts, Counter())
        os.remove(p)
        random.Random(SEED).shuffle(v)
        val = v[:n] + val                     # NPTEL first, then LibriSpeech
    write_arrow(pa.table({c: [r[c] for r in val] for c in COLS}), VAL_DIR / "val.arrow")
    n_val = Counter(r["source"] for r in val)
    del v, val; gc.collect()

    reasons = {"nptel": Counter(), "libri": Counter()}

    # -- LibriSpeech train.clean.100 (American English share): shards spread across the speaker list
    libri_files = repo_files(LIBRI_REPO, "clean/train.100/")
    libri_queue = sorted(set(manifest.loc[manifest.source == "libri", "shard"])) if rebuild else \
        [libri_files[i] for i in np.linspace(0, len(libri_files) - 1, LIBRI_SHARDS).astype(int)]
    libri_meta, _ = collect(LIBRI_REPO, libri_queue, "text", "speaker_id", "id", "libri",
                            float("inf"), reasons["libri"], keep, None, None)

    # -- NPTEL train shards in random order until we have enough clean candidates
    nptel_queue = sorted(set(manifest.loc[manifest.source == "nptel", "shard"])) if rebuild else \
        random.Random(SEED).sample(repo_files(NPTEL_REPO, "data/train-"), k=len(repo_files(NPTEL_REPO, "data/train-")))
    wps_all = []
    nptel_meta, stopped_early = collect(NPTEL_REPO, nptel_queue, "transcription_normalised", "speaker_name",
                                        "file_name", "nptel", NPTEL_HOURS * CANDIDATE_FACTOR,
                                        reasons["nptel"], keep, wps_all, NPTEL_STOP_MIN)

    if rebuild:
        nptel_sel, libri_sel = nptel_meta, libri_meta
        band = json.loads(PREP_REPORT.read_text()).get("nptel_wps_band")
    else:
        # Tighter NPTEL filter: drop speaking rates far from this dataset's own typical range
        # (median +/- WPS_ROBUST_K robust standard deviations, kept inside the fixed limits).
        w = np.array(wps_all)
        med = float(np.median(w)); rsd = float(np.median(np.abs(w - med)) * 1.4826)
        band = [round(max(MIN_WPS, med - WPS_ROBUST_K * rsd), 3), round(min(MAX_WPS, med + WPS_ROBUST_K * rsd), 3)]
        outside = ~nptel_meta.wps.between(*band)
        reasons["nptel"]["words_per_second_dataset_band"] = int(outside.sum())
        reasons["nptel"]["passed_fixed_filters"] -= int(outside.sum())
        nptel_meta = nptel_meta[~outside]
        got_h = nptel_meta.duration.sum() / 3600
        nptel_sel = round_robin(nptel_meta, NPTEL_HOURS)
        # keep American English at LIBRI_SHARE of the mix even if NPTEL came up short
        libri_h = min(LIBRI_HOURS, nptel_sel.duration.sum() / 3600 * LIBRI_SHARE / (1 - LIBRI_SHARE))
        libri_sel = round_robin(libri_meta, libri_h)
        print(f"NPTEL speaking-rate band {band[0]:.2f} to {band[1]:.2f} words/s "
              f"(median {med:.2f}); {got_h:.1f} h remain for selection")

    # -- Final training files (Arrow, memory-mapped by datasets), candidates deleted as we go
    sel_ids = set(nptel_sel.file_name) | set(libri_sel.file_name)
    for c in sorted(CAND.glob("*.parquet")):
        t = pq.read_table(c)
        t = t.filter(pa.array([x in sel_ids for x in t.column("file_name").to_pylist()]))
        if t.num_rows:
            write_arrow(t, TRAIN_DIR / c.name.replace(".parquet", ".arrow"))
        c.unlink()

    # -- Manifest and report on Drive
    if not rebuild:
        pd.concat([nptel_sel, libri_sel]).to_csv(MANIFEST, index=False)
    sel = pd.read_csv(MANIFEST, dtype={"file_name": str})
    rn = reasons["nptel"]
    wps_removed = rn["words_per_second_fixed"] + rn["words_per_second_dataset_band"]
    report = {
        "nptel_hours": round(sel[sel.source == "nptel"].duration.sum() / 3600, 2),
        "nptel_speakers": int(sel[sel.source == "nptel"].speaker.nunique()),
        "nptel_clips": int((sel.source == "nptel").sum()),
        "nptel_shards_used": int(sel[sel.source == "nptel"].shard.nunique()),
        "libri_hours": round(sel[sel.source == "libri"].duration.sum() / 3600, 2),
        "libri_speakers": int(sel[sel.source == "libri"].speaker.nunique()),
        "libri_clips": int((sel.source == "libri").sum()),
        "libri_share_of_hours": round(sel[sel.source == "libri"].duration.sum() / sel.duration.sum(), 3),
        "nptel_wps_band": band,
        "nptel_clips_read": rn["read"],
        "nptel_pct_removed_by_speaking_rate": round(100 * wps_removed / max(rn["read"], 1), 2),
        "nptel_pct_removed_by_all_filters": round(100 * (rn["read"] - rn["passed_fixed_filters"]) / max(rn["read"], 1), 2),
        "stopped_early_on_time": stopped_early,
        "filter_counts_nptel": dict(rn),
        "filter_counts_libri": dict(reasons["libri"]),
        "val_clips": dict(n_val),
        "licenses": {"NPTEL2020": "CC BY 4.0 (Zenodo records)", "LibriSpeech": "CC BY 4.0",
                     "Svarah": "CC BY 4.0"},
    }
    if not rebuild:
        PREP_REPORT.write_text(json.dumps(report, indent=1))
    print(json.dumps(report, indent=1))
    print(f"Prep took {(time.time() - t_start) / 60:.1f} min; local disk free: "
          f"{shutil.disk_usage(str(WORK)).free / 1e9:.0f} GB")

## Stage 2. Baseline testing (budget 20 min)

We run the unchanged `whisper-tiny` in two ways:
- **Paper path:** `openai-whisper` `transcribe()` with the same settings as `whisper_final.py`, with
  the random seed fixed for each clip. This seeded rerun is our official "before" number. The
  paper's own CSV (4.52% American, 8.86% Indian) is shown as a reference row.
- **Transformers path:** the Hugging Face model with English forced, which is the code path the
  fine-tuned model is trained in.

Cell 7 runs the transformers path on Svarah. Both cells skip work whose CSV already exists on Drive.

In [ ]:
# Cell 6. Baseline on test set A, both paths
from transformers import WhisperForConditionalGeneration, WhisperProcessor

processor = WhisperProcessor.from_pretrained(MODEL_ID, language="english", task="transcribe")
with stage("2 baseline"):
    out = RESULTS / "testA_baseline_openai.csv"
    if not out.exists():
        transcribe_openai(whisper.load_model("tiny", device=DEVICE), "tiny-baseline-openai").to_csv(out, index=False)
    out = RESULTS / "testA_baseline_hf.csv"
    if not out.exists():
        base_hf = WhisperForConditionalGeneration.from_pretrained(MODEL_ID).to(DEVICE).eval()
        transcribe_hf_test_a(make_asr_pipeline(base_hf, processor), "tiny-baseline-hf").to_csv(out, index=False)

rows = [summarise(load_scored(PAPER_CSV, spelled=s), f"paper CSV{' spelled' if s else ''}") for s in (False, True)]
rows += [summarise(load_scored(RESULTS / f"testA_baseline_{p}.csv", spelled=s), f"baseline {p}{' spelled' if s else ''}")
         for p in ("openai", "hf") for s in (False, True)]
show_summary(rows)
print("\n" + baseline_note())

In [ ]:
# Cell 7. Baseline on Svarah (test set B), transformers path
with stage("2 baseline"):
    base_hf = WhisperForConditionalGeneration.from_pretrained(MODEL_ID).to(DEVICE).eval()
    sv_base = load_scored(run_svarah(base_hf, "tiny-baseline-hf", RESULTS / "svarah_baseline_hf.csv"))
print(f"Svarah baseline: {len(sv_base)} clips, mean per-clip WER {sv_base.wer.mean() * 100:.2f}%, "
      f"corpus WER {jiwer.wer(list(sv_base.ref), list(sv_base.hyp)) * 100:.2f}%")
print(sv_base[["ground_truth", "prediction"]].head(5).to_string())

## Stage 3. Training (budget 7 hours)

Full fine-tune of whisper-tiny in fp16: learning rate 1e-5, warmup 500 steps, batch size 32, linear decay.
- **Cell 8** loads the data and defines the trainer. A clip's audio is decoded only when its batch is built.
- **Cell 9** times the first 50 steps and one validation pass, then sets the total step count to fit
  7 hours, capped at `MAX_EPOCHS`. It saves the plan to Drive and is skipped on reruns.
- **Cell 10** trains. Every 30 minutes, and every 1,000 steps, it scores the held-out validation clips
  (200 NPTEL, 100 LibriSpeech) and saves a checkpoint to Drive. We keep only the latest checkpoint and
  the one with the lowest mean validation WER. Training stops at the step budget or the 7-hour budget,
  whichever comes first. Rerunning the cell resumes from the latest checkpoint. We never use test sets
  A and B to choose a checkpoint.

In [ ]:
# Cell 8. Datasets, collator, metrics, trainer factory
import datasets, inspect
from dataclasses import dataclass
from transformers import (Seq2SeqTrainer, Seq2SeqTrainingArguments, TrainerCallback,
                          WhisperForConditionalGeneration, WhisperProcessor)
from transformers.trainer_utils import get_last_checkpoint

processor = WhisperProcessor.from_pretrained(MODEL_ID, language="english", task="transcribe")
tok = processor.tokenizer
train_ds = datasets.concatenate_datasets(
    [datasets.Dataset.from_file(str(f)) for f in sorted(TRAIN_DIR.glob("*.arrow"))]
).select_columns(["audio_bytes", "text"])
val_ds = datasets.Dataset.from_file(str(VAL_DIR / "val.arrow"))
N_VAL_NP = sum(s == "nptel" for s in val_ds["source"])
val_ds = val_ds.select_columns(["audio_bytes", "text"])
print(f"Train clips: {len(train_ds)}, validation clips: {len(val_ds)} ({N_VAL_NP} NPTEL first)")

def new_model():
    m = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
    m.generation_config.language = "en"
    m.generation_config.task = "transcribe"
    m.generation_config.forced_decoder_ids = None
    m.config.forced_decoder_ids = None
    m.config.use_cache = False
    return m

@dataclass
class Collator:
    processor: WhisperProcessor
    decoder_start_token_id: int
    def __call__(self, feats):
        audios = [decode_audio(f["audio_bytes"]) for f in feats]
        batch = self.processor.feature_extractor(audios, sampling_rate=16000, return_tensors="pt")
        labels = self.processor.tokenizer([f["text"] for f in feats], padding=True, return_tensors="pt")
        ids = labels["input_ids"].masked_fill(labels["attention_mask"].ne(1), -100)
        if (ids[:, 0] == self.decoder_start_token_id).all():
            ids = ids[:, 1:]
        return {"input_features": batch["input_features"], "labels": ids}

collator = Collator(processor, new_model().config.decoder_start_token_id)

def compute_metrics(pred):
    p, l = pred.predictions.copy(), pred.label_ids.copy()
    p[p == -100] = tok.pad_token_id
    l[l == -100] = tok.pad_token_id
    hyp = [normalise(x) for x in tok.batch_decode(p, skip_special_tokens=True)]
    ref = [normalise(x) for x in tok.batch_decode(l, skip_special_tokens=True)]
    w_np = jiwer.wer(ref[:N_VAL_NP], hyp[:N_VAL_NP])
    w_li = jiwer.wer(ref[N_VAL_NP:], hyp[N_VAL_NP:])
    return {"wer_nptel": w_np * 100, "wer_libri": w_li * 100, "wer_mean": (w_np + w_li) / 2 * 100}

def make_args(output_dir, max_steps):
    return Seq2SeqTrainingArguments(
        output_dir=str(output_dir), max_steps=max_steps,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=1, learning_rate=LR, warmup_steps=WARMUP_STEPS,
        lr_scheduler_type="linear", fp16=(DEVICE == "cuda"),
        eval_strategy="no", save_strategy="no", logging_steps=25, report_to="none",
        predict_with_generate=True, generation_max_length=225,
        dataloader_num_workers=2, remove_unused_columns=False, seed=SEED, data_seed=SEED)

def make_trainer(model, args, callbacks=()):
    kw = dict(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
              data_collator=collator, compute_metrics=compute_metrics, callbacks=list(callbacks))
    key = "processing_class" if "processing_class" in inspect.signature(Seq2SeqTrainer.__init__).parameters else "tokenizer"
    kw[key] = processor.feature_extractor
    return Seq2SeqTrainer(**kw)

TRAIN_STATE = DRIVE_DIR / "train_state.json"
RUN_CFG = DRIVE_DIR / "run_config.json"

class BudgetCheckpointCallback(TrainerCallback):
    """Evaluates and saves every SAVE_EVERY_MIN minutes or EVAL_EVERY_STEPS steps, keeps the best
    and the latest checkpoint, and stops at the training time budget (counted across sessions)."""
    def __init__(self):
        st = json.loads(TRAIN_STATE.read_text()) if TRAIN_STATE.exists() else {}
        self.prev_s = st.get("train_seconds", 0.0)
        self.best = st.get("best")
        self.evals = st.get("evals", [])
        self.pending = None
    def used(self):
        return self.prev_s + time.time() - self.t0
    def on_train_begin(self, args, state, control, **kw):
        self.t0 = self.last = time.time()
    def on_step_end(self, args, state, control, **kw):
        due = (time.time() - self.last >= SAVE_EVERY_MIN * 60
               or state.global_step % EVAL_EVERY_STEPS == 0
               or state.global_step >= state.max_steps)
        if self.used() >= TRAIN_BUDGET_H * 3600:
            print("Training time budget reached: saving and stopping.")
            control.should_training_stop = True
            due = True
        if due:
            control.should_evaluate = True
            control.should_save = True
            self.last = time.time()
        return control
    def on_evaluate(self, args, state, control, metrics=None, **kw):
        m = {k: round(v, 3) for k, v in (metrics or {}).items() if k.startswith("eval_wer")}
        self.pending = (state.global_step, m.get("eval_wer_mean"))
        self.evals.append({"step": state.global_step, "train_minutes": round(self.used() / 60, 1), **m})
        print(f"step {state.global_step}: {m}")
    def on_save(self, args, state, control, **kw):
        ckpt = Path(args.output_dir) / f"checkpoint-{state.global_step}"
        if self.pending and self.pending[0] == state.global_step and self.pending[1] is not None \
                and (self.best is None or self.pending[1] < self.best["wer_mean"]):
            self.best = {"step": state.global_step, "wer_mean": self.pending[1], "dir": str(ckpt)}
        keep = {ckpt.name, Path(self.best["dir"]).name if self.best else ""}
        for d in Path(args.output_dir).glob("checkpoint-*"):
            if d.name not in keep:
                shutil.rmtree(d, ignore_errors=True)
        TRAIN_STATE.write_text(json.dumps({"train_seconds": self.used(), "best": self.best,
                                           "evals": self.evals, "global_step": state.global_step}, indent=1))
print("Trainer ready.")

In [ ]:
# Cell 9. Time the first 50 steps and one validation pass, then fix the step count (runs once)
if RUN_CFG.exists():
    cfg = json.loads(RUN_CFG.read_text())
    print("Using saved plan:", cfg)
else:
    class StepTimer(TrainerCallback):
        def on_step_end(self, args, state, control, **kw):
            if state.global_step == 1: self.t1 = time.time()
            if state.global_step == 51: self.t51 = time.time()
    with stage("3 training"):
        timer = StepTimer()
        bench = make_trainer(new_model(), make_args(WORK / "bench", 51), [timer])
        bench.train()
        sec_per_step = (timer.t51 - timer.t1) / 50
        t = time.time(); bench.evaluate(); eval_s = time.time() - t
        del bench; gc.collect(); torch.cuda.empty_cache()
    save_s = 45                                       # one checkpoint written to Drive
    check_s = eval_s + save_s
    usable_s = TRAIN_BUDGET_H * 3600 - 15 * 60        # 15 min margin for loading and the final save
    usable_s *= 1 - check_s / (SAVE_EVERY_MIN * 60)   # time spent on the 30-minute checkpoints
    fit_steps = int(usable_s / (sec_per_step + check_s / EVAL_EVERY_STEPS))
    steps_per_epoch = math.ceil(len(train_ds) / BATCH_SIZE)
    cfg = {"sec_per_step": round(sec_per_step, 3), "eval_seconds": round(eval_s, 1),
           "steps_per_epoch": steps_per_epoch, "steps_that_fit_budget": fit_steps,
           "epoch_cap_steps": MAX_EPOCHS * steps_per_epoch,
           "max_steps": min(fit_steps, MAX_EPOCHS * steps_per_epoch)}
    cfg["expected_train_hours"] = round(cfg["max_steps"] * (sec_per_step + check_s / EVAL_EVERY_STEPS) / 3600, 2)
    cfg["epochs"] = round(cfg["max_steps"] / steps_per_epoch, 2)
    RUN_CFG.write_text(json.dumps(cfg, indent=1))
    print(json.dumps(cfg, indent=1))

In [ ]:
# Cell 10. Train (rerun this cell after a disconnect to resume)
cfg = json.loads(RUN_CFG.read_text())
cb = BudgetCheckpointCallback()
last = get_last_checkpoint(str(CKPT_DIR)) if any(CKPT_DIR.glob("checkpoint-*")) else None
done_steps = int(last.rsplit("-", 1)[1]) if last else 0
if cb.prev_s >= TRAIN_BUDGET_H * 3600 or done_steps >= cfg["max_steps"]:
    print(f"Training already finished at step {done_steps}. Best checkpoint: {cb.best}")
else:
    print(f"{'Resuming from ' + last if last else 'Starting'}; target {cfg['max_steps']} steps, "
          f"{cb.prev_s / 3600:.2f} of {TRAIN_BUDGET_H} h used so far")
    trainer = make_trainer(new_model(), make_args(CKPT_DIR, cfg["max_steps"]), [cb])
    with stage("3 training"):
        trainer.train(resume_from_checkpoint=last)
    pd.DataFrame(trainer.state.log_history).to_csv(RESULTS / "training_log.csv", index=False)
    print("Best checkpoint:", cb.best)

## Stage 4. Final testing (budget 60 min)

Cell 11 exports the best checkpoint to Drive in both Hugging Face and openai-whisper formats. Cell 12
runs the fine-tuned model on test sets A and B with the same two paths as the baseline. Cell 13 builds
the comparison tables, runs the key checks, and writes `send_back.zip` to Drive.

In [ ]:
# Cell 11. Export the best checkpoint (both formats)
best = json.loads(TRAIN_STATE.read_text())["best"]
print("Best checkpoint:", best)
with stage("4 final testing"):
    ft = WhisperForConditionalGeneration.from_pretrained(best["dir"])
    ft.generation_config.forced_decoder_ids = None
    ft.save_pretrained(FINAL_DIR); processor.save_pretrained(FINAL_DIR)
    dims = whisper.load_model("tiny", device="cpu").dims
    OPENAI_CKPT = FINAL_DIR / "whisper_tiny_indian_ft_openai.pt"
    torch.save({"dims": dims.__dict__, "model_state_dict": hf_to_openai(ft.state_dict())}, OPENAI_CKPT)
    whisper.load_model(str(OPENAI_CKPT), device="cpu")   # fails loudly if any weight is missing
print("Saved to", FINAL_DIR)

In [ ]:
# Cell 12. Fine-tuned model on test sets A and B
with stage("4 final testing"):
    out = RESULTS / "testA_finetuned_openai.csv"
    if not out.exists():
        transcribe_openai(whisper.load_model(str(OPENAI_CKPT), device=DEVICE), "tiny-finetuned-openai").to_csv(out, index=False)
    ft_hf = WhisperForConditionalGeneration.from_pretrained(FINAL_DIR).to(DEVICE).eval()
    out = RESULTS / "testA_finetuned_hf.csv"
    if not out.exists():
        transcribe_hf_test_a(make_asr_pipeline(ft_hf, processor), "tiny-finetuned-hf").to_csv(out, index=False)
    run_svarah(ft_hf, "tiny-finetuned-hf", RESULTS / "svarah_finetuned_hf.csv")
print("Done.")

In [ ]:
# Cell 13. Comparison tables, key checks, and the zip to send back
with stage("4 final testing"):
    # ---- test set A: every model x path x numeral handling
    rows, scored = [summarise(load_scored(PAPER_CSV), "paper CSV")], {}
    for model in ("baseline", "finetuned"):
        for path in ("openai", "hf"):
            for spelled in (False, True):
                name = f"{model} {path}{' spelled' if spelled else ''}"
                scored[name] = load_scored(RESULTS / f"testA_{model}_{path}.csv", spelled=spelled)
                rows.append(summarise(scored[name], name))
    summary_a = pd.DataFrame(rows)
    summary_a.to_csv(RESULTS / "testA_summary.csv", index=False)

    # paired before/after test per group (same speakers): one-sided Wilcoxon, fine-tuned lower
    paired = []
    for path in ("openai", "hf"):
        for spelled in ("", " spelled"):
            b, f = scored[f"baseline {path}{spelled}"], scored[f"finetuned {path}{spelled}"]
            for g in ("US", "India"):
                x, y = b[b.group == g].wer.to_numpy(), f[f.group == g].wer.to_numpy()
                p_lower = stats.wilcoxon(y, x, alternative="less").pvalue if np.any(x != y) else 1.0
                p_higher = stats.wilcoxon(y, x, alternative="greater").pvalue if np.any(x != y) else 1.0
                paired.append({"path": path + spelled, "group": g, "before_pct": x.mean() * 100,
                               "after_pct": y.mean() * 100, "change_pp": (y.mean() - x.mean()) * 100,
                               "clips_better": int((y < x).sum()), "clips_worse": int((y > x).sum()),
                               "p_after_lower": p_lower, "p_after_higher": p_higher})
    paired = pd.DataFrame(paired)
    paired.to_csv(RESULTS / "testA_before_after.csv", index=False)

    # substitution pairs before and after (paper path, raw scoring)
    sub_rows = []
    for model in ("baseline", "finetuned"):
        c = substitution_pairs(scored[f"{model} openai"])
        for g in ("US", "India"):
            for (r, h), n in c[g].items():
                sub_rows.append({"model": model, "group": g, "reference_word": r, "transcribed_as": h, "count": n})
    subs = pd.DataFrame(sub_rows)
    subs.to_csv(RESULTS / "testA_substitutions.csv", index=False)
    focus = (subs[subs.reference_word.isin(["slabs", "snake", "frog", "snack", "peas", "spoons", "bags", "scoop"])]
             .pivot_table(index=["group", "reference_word", "transcribed_as"], columns="model",
                          values="count", aggfunc="sum", fill_value=0))

    # ---- test set B: Svarah, Indian only, paired before/after
    sb = load_scored(RESULTS / "svarah_baseline_hf.csv")
    sf_ = load_scored(RESULTS / "svarah_finetuned_hf.csv")
    sv_rows = [{"condition": n, "clips": len(d), "mean_wer_pct": d.wer.mean() * 100,
                "se_pct": d.wer.std(ddof=1) / np.sqrt(len(d)) * 100,
                "corpus_wer_pct": jiwer.wer(list(d.ref), list(d.hyp)) * 100,
                "subs": int(d.subs.sum()), "dels": int(d.dels.sum()), "ins": int(d.ins.sum())}
               for n, d in (("baseline", sb), ("finetuned", sf_))]
    sv_p = stats.wilcoxon(sf_.wer, sb.wer, alternative="less").pvalue
    summary_b = pd.DataFrame(sv_rows); summary_b["p_after_lower"] = [np.nan, sv_p]
    summary_b.to_csv(RESULTS / "svarah_summary.csv", index=False)
    group_cols = [c for c in ("primary_language", "gender", "age-group", "age_group") if c in sb.columns]
    for c in group_cols:
        by = pd.DataFrame({"baseline": sb.groupby(c).wer.mean() * 100, "finetuned": sf_.groupby(c).wer.mean() * 100,
                           "clips": sb.groupby(c).size()})
        by.to_csv(RESULTS / f"svarah_by_{c}.csv")

    # ---- copy logs and zip
    for f in (TIME_LOG, RUN_CFG, TRAIN_STATE, DRIVE_DIR / "prep_report.json"):
        if f.exists(): shutil.copy(f, RESULTS / f.name)
    budget_report().to_csv(RESULTS / "time_budget.csv", index=False)
    note = baseline_note()
    (RESULTS / "baseline_note.txt").write_text(note + "\n")

zip_path = DRIVE_DIR / "send_back.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(RESULTS.iterdir()):
        z.write(f, f.name)

# ---- printout
print("\nTest set A (mean per-clip WER %, n = 60 per group):")
show_summary(rows)
print("\nBefore vs after, per group:")
print(paired.round(4).to_string(index=False))
r0, r1 = summary_a.set_index("condition").loc["baseline openai"], summary_a.set_index("condition").loc["finetuned openai"]
print("\n" + note)
print("\nKey checks (paper path; 'before' is the seeded rerun):")
print(f"  Indian WER fell:        {r0.india_mean_pct:.2f}% -> {r1.india_mean_pct:.2f}%")
print(f"  US WER flat or better:  {r0.us_mean_pct:.2f}% -> {r1.us_mean_pct:.2f}%")
print(f"  Ratio shrank:           {r0.ratio:.2f}x -> {r1.ratio:.2f}x")
print(f"  Insertions (US/Indian): {r0.us_ins_total}/{r0.india_ins_total} -> {r1.us_ins_total}/{r1.india_ins_total}")
print(f"  Clips with digits (Indian): {r0.india_clips_with_digits} -> {r1.india_clips_with_digits}")
print("\nSelected substitution pairs (paper path):")
print(focus.to_string() if len(focus) else "  none")
print("\nSvarah (test set B):")
print(summary_b.round(4).to_string(index=False))
print(f"\nSend back: {zip_path}")